# Lab 4 — Clean the broken sensor feed
### EP0418 · Topic 4 · 2 practical hours

**You need:** `ep0418.py` and `smart_building_sensors.csv` beside this notebook.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

Every prompt in these notebooks opens with the rules that matter for that task:
do not drop rows, show the row count before and after, treat -999 as missing,
explain the code in plain words. Keep those lines when you adapt a prompt, and
start your own prompts the same way.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

> ### This dataset is deliberately broken
> **Six** separate faults are hidden in this file. The summary table in the
> next two cells shows three of them, if you read it carefully. The checks catch
> three. **Two you will only find by plotting** — and I am telling you that up
> front so you keep looking.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True


def submit_prediction(team, lab, prediction, method):
    """Send one team's number to the Prediction League board."""
    import urllib.parse
    form = "https://docs.google.com/forms/d/e/1FAIpQLSdwpzHafC_rmPDdbK_cm6J8CihD-NSVS7ZBUyYCWIrS_MMJ0g/formResponse"
    fields = {'team': 'entry.911171418', 'lab': 'entry.1890998159', 'prediction': 'entry.959891957', 'method': 'entry.1997288724'}
    if not team or not method or prediction is None:
        print("Fill in TEAM, MY_PREDICTION and METHOD above, then run this cell again."); return
    entry = {"team": team, "lab": str(lab), "prediction": f"{float(prediction):.1f}", "method": method}
    if not form:
        print("The Prediction League is not switched on yet; tell your tutor your number."); return
    data = urllib.parse.urlencode({fields[k]: v for k, v in entry.items()}).encode()
    try:
        urllib.request.urlopen(urllib.request.Request(form, data=data, headers={"User-Agent": "EP0418"}), timeout=60)
        print(f"Entered: {team} predicts {float(prediction):.1f}. Check the scoreboard in a minute.")
    except Exception as e:
        print("The league did not answer:", e, "- try again in a minute.")

CSV = "smart_building_sensors.csv"
fetch(CSV)

raw = pd.read_csv(CSV, parse_dates=["timestamp"])
print(f"{len(raw):,} rows, {raw.timestamp.min()} -> {raw.timestamp.max()}")

In [ ]:
print("blank values:", raw.isna().sum().sum())
print("duplicated timestamps:", raw.timestamp.duplicated().sum())
print()
print(raw.describe().T[["min", "mean", "max"]].round(1).to_string())

### Read that table before going further

`isna()` reports zero blanks. But look at the minimum for `humidity_pct`.

**−999 is not a reading.** It is the logger saying "no answer". pandas counts it
as a real number, so it is dragging your mean down right now and `isna()` will
never find it.

How many *other* columns have a min or max that is physically impossible?
Compare each maximum with its mean: a classroom does not use eight times its
usual energy in five minutes.

## Task 1 — Replace the sentinels

> I have a DataFrame `raw` where −999 in `humidity_pct` means "no reading".
> Replace those with NaN. Do not remove any rows.

Read what it gives you. A common wrong answer filters the rows out entirely
instead of marking them blank — which changes your row count, and the contract
below will catch it.

In [ ]:
# TODO(you): `clean` = copy of raw with -999 in humidity_pct replaced by NaN
clean = ...

# ---- verification contract — do not edit ----
check.no_sentinels(clean, cols=["humidity_pct"], sentinel=-999)
check.rows_at_least(clean, 17000)
check.report()

**Expected output** *(computed from the data file when this notebook was built)*: still 17,354 rows; 95 values of −999 became NaN. If the row count fell, rows were dropped rather than values replaced.

**Look for** what changed between the two panels, and what did not.

In [ ]:
if filled(clean):
    show.before_after(raw, clean, "humidity_pct", title="Replacing -999 with NaN")

**What you should see:** the top panel has the vertical spikes to −999; the bottom
panel has small gaps in the line where they were. The row count is the same in
both titles. Nothing was removed: the failures are now shown as *missing* instead
of as a number, which is what they always were.

## Task 1b — The same request, with and without the course instructions (10 min)

Your Gemini follows the course instructions in this notebook. Most AI tools you
will meet at work will not. This shows what the instructions were doing for you.

1. Open a **new** notebook (File > New notebook in Drive). It has no custom
   instructions. Open its Gemini panel and ask exactly:
   > Clean the humidity column in my data. The column is humidity_pct and some readings are -999.
2. Ask the Gemini in **this** notebook the same thing, word for word.
3. Compare the two answers:

| Question | Without instructions | With instructions |
|---|---|---|
| Did its code remove any rows? | | |
| Did it show the row count before and after? | | |
| Did it say what −999 means? | | |
| Did it explain its code in plain words? | | |

**Without the instructions, what would you have had to check yourself?**

The instructions are a seatbelt, not a driver. With any AI tool, the checks are
still yours.

## Task 2 — Duplicates and time order

Decide: keep the first, keep the last, or average them? On a logger that retries,
the last is usually the corrected value. Whatever you choose, be ready to justify
it in your log.

In [ ]:
# TODO(you): drop duplicate timestamps and sort into time order
clean = ...

# ---- verification contract — do not edit ----
check.no_duplicate_timestamps(clean, col="timestamp")
check.sorted_in_time(clean, col="timestamp")
check.report()

**Expected output** *(computed from the data file when this notebook was built)*: 17,354 rows become 17,214: 140 duplicate timestamps removed. Write which copy you kept and why in your Decision line.

## Task 3 — Put it on a regular grid

In [ ]:
# TODO(you): resample `clean` onto a regular 5-minute grid called `grid`
grid = ...

# ---- verification contract — do not edit ----
check.regular_cadence(grid, "5min")
check.values_in_range(grid.reset_index(), "temp_c", 15, 40)
check.report()

**Expected output** *(computed from the data file when this notebook was built)*: 17,214 rows become 17,280 on the grid: the count goes *up* by 66, the empty slots of the 11 June outage, which now exist as rows of NaN.

**Look for** the shaded hours on 11 June.

In [ ]:
if filled(grid):
    show.series(grid, col="co2_ppm", title="CO2 on the regular 5-minute grid; shaded: the 11 June outage",
                span=("2026-06-11 02:00", "2026-06-11 07:30"))

**What you should see:** a break in the line inside the shaded period. Before
resampling, those hours simply had no rows and nothing showed. Now they are empty
slots on the grid, and the chart is honest about them. Count them in Task 4: the
row count goes *up* from `clean` to `grid`, and you have to say why.

## Task 4 — Account for every row you lost

Your row count changed three times. Explain why, in your own words.

*"Cleaning removed some rows"* will be rejected. Say **which** rows and **why**.

In [ ]:
# TODO(you): explain, in your own words, every change in the row count
EXPLANATION = "TODO(you): replace with a real explanation"

# ---- verification contract — do not edit ----
check.rows_accounted_for(
    raw, clean, grid,
    names=["raw", "clean", "grid"],
    explain=EXPLANATION,
)
check.report()

## Task 5 — Find the two faults no check can catch

Plot all four sensor channels. Sixty days on one axis hides a fourteen-hour
flat line, so also plot a few days at a time, or ask your assistant for the
longest run of identical consecutive readings in each channel and the date it
starts.

One fault is a sensor that **stops changing** — it produces no outliers at all,
just a flat line, which looks perfectly well-behaved to any alarm.

The other is a **slow drift** that makes a fixed threshold fire more and more
often even though the room never changed.

In [ ]:
# Draw all four channels. The finding is yours; the drawing is not the exercise.
if filled(grid):
    for col in ["temp_c", "humidity_pct", "co2_ppm", "energy_kwh"]:
        show.series(grid, col=col, title=col)

**Look for:** in one channel, a stretch that is perfectly flat; in another, a slow
climb over the last three weeks that never comes back down. Neither is out of
range, neither is a duplicate, neither is missing. Every check in the library
passes on both. Write down which channel, which dates, and what you think the
sensor was doing.

---
## Predict: How many bone-dry days next week? (10 minutes, teams of two or three)

*The Prediction League, week 3. Same team name as last time.*

On how many of the seven days of the target week will the island-average rain be zero? Lab 4 was about zero versus missing; this is the zero. The baseline, doing nothing clever, is on the board already:
the share of dry days in the sixty-day file, times seven, rounded. The target week's dates are on the scoreboard, and last week's
result is there too: look at who was closest and what they said they did.

In [ ]:
fetch("sg_rainfall_daily.csv")
rain = pd.read_csv("sg_rainfall_daily.csv", parse_dates=["date"])
dry = rain.rain_mm_mean == 0
print(f"dry days in the sixty-day file : {int(dry.sum())} of {len(rain)}")
print(f"share x 7, rounded             : {round(dry.mean() * 7)} days   <- the baseline")
print(f"dry days in the last fortnight : {int(dry.tail(14).sum())}")

**Step by step.** Open Gemini (spark icon, bottom right), **+** for a new chat,
paste, Enter:

> I am new to this. Answer in three plain sentences, no headings, no code.
> On how many of the seven days of the target week will the island-average rain be zero? What would you base the guess on, using what this week's lab taught,
> and why might it still be wrong?

Read the three sentences. Decide with your team: the baseline as printed, one of
the other numbers printed above, or a number Gemini computes for you (rules first,
**+ Code**, paste, run, read). Fill in the three lines below and run the cell.

In [ ]:
TEAM = ""                 # TODO(you): the same team name as last week
MY_PREDICTION = None      # TODO(you): your number, e.g. 1
METHOD = ""               # TODO(you): one line on how you got it

submit_prediction(TEAM, 4, MY_PREDICTION, METHOD)

---
## Read before you run

An assistant was asked to resample the cleaned sensor data onto a 5-minute grid so that `no_missing` passes. It wrote:

The code is in the cell below, commented out. It came from an AI assistant in
someone else's chat; you are not asking Gemini for it. **Do not run it yet.** Read
it, answer the two questions below, and only then remove the `#` signs and run it
to see whether you were right.

What does the second line do to the 5½-hour outage on 11 June? Will any of the contracts in this lab fail?

Write down how many readings the second line invents.

<details><summary>Hint 1: where to look</summary>

What does `interpolate()` do to a run of empty slots between two real readings?

</details>
<details><summary>Hint 2: closer to the answer</summary>

It draws a straight line through them: 66 five-minute readings for the outage and 159 humidity blanks are invented. The row count does not change, so `rows_accounted_for` cannot see it.

</details>

**After you have written your answer, and not before:** select the cell, click the
pencil-and-spark icon at its top right and choose **Explain code**. Did Gemini notice
the fault? It often does not. That is the point: an explanation that reads well is
not evidence the code is right.

**To fix it:** choose **Transform code** on the same cell and say what to change, in
plain words. Colab shows the change as a diff before it is applied. Read the diff. If
it touches a `check.` line, reject it.

In [ ]:
# An AI assistant wrote this for a classmate. It is already here: there is nothing to ask Gemini for.
# 1. Read it. Answer the two questions in the cell below, in your own words.
# 2. Then select this cell, click the pencil-and-spark icon, choose Explain code. Did Gemini see the fault?
# 3. Only then remove the # signs and run it, to see whether you were right.
# grid = clean.set_index("timestamp").resample("5min").mean()
# grid = grid.interpolate()

**What is wrong with it:**

**What it would produce, and what it should produce:**

---
## Quiz yourself before you leave (5 minutes)

The checks tell you whether the code is right. They do not tell you whether
**you** know what it did. Paste this into the Gemini chat:

> Quiz me on this lab. Ask five short questions, one at a time, in plain words, on these ideas: the four disguises of a missing value; why duplicate timestamps appear and which copy to keep; why every row you remove must be accounted for; what rows_accounted_for checks and what it cannot see; what a stuck sensor looks like and why no check catches it. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks in the chat panel; you answer in the chat box; it marks you and asks
the next. Ignore the suggestion chips under its messages. Answer in your own
words, without looking back until it has marked you.

Then write your score and the idea to revise below. Your tutor asks for them in
the plenary.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

---
## Before you leave

Below: one **mistake caught** line per AI mistake you caught today, and
one **decision** line for each fault you handled: what you did about the duplicates,
the −999s and the outage gap, and why.

Then answer: **which date is the stuck sensor on, and which channel?** Bring the
answer to the next session.

**Mistake caught:** *(what I expected · what was there · how I found it)*

**Decision:** *(what I decided · why · the evidence)*